# ADP PM Notebook 2 — RAG Build, Evaluation and Improvement

**Purpose:** Combine the original RAG fundamentals notebook and the learner RAG activity into one shorter notebook.

**Retained:** S3 document loading, chunking, embeddings, retrieval, grounded answer generation, evidence inspection, golden-question evaluation.



## PM learning goal

By the end, learners should be able to explain:

> RAG connects an LLM to trusted enterprise documents so answers can be grounded in retrieved evidence.

The key PM question is not only “Did the model answer?” but also “Did it retrieve the right evidence?”


In [ ]:
# Uncomment only if needed.
%pip install -q boto3 pandas numpy python-docx ipykernel


In [ ]:
from __future__ import annotations

import io
import os
import re
import json
import time
import boto3
import numpy as np
import pandas as pd
from docx import Document as DocxDocument
from IPython.display import display

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
BEDROCK_LLM_MODEL_ID = "amazon.nova-lite-v1:0"
BEDROCK_EMBEDDING_MODEL_ID = "amazon.titan-embed-text-v1"

# Change these values for the classroom bucket.
S3_BUCKET = "rag-demo-docs-sri"
S3_PREFIX = "rag-docs/"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 120
TOP_K = 4
MAX_CHUNKS_TO_EMBED = None  # Set to a small number during testing if needed.

s3 = boto3.client("s3", region_name=AWS_REGION)
bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("AWS Region:", AWS_REGION)
print("LLM model:", BEDROCK_LLM_MODEL_ID)
print("Embedding model:", BEDROCK_EMBEDDING_MODEL_ID)
print("S3 path:", f"s3://{S3_BUCKET}/{S3_PREFIX}")


## 1. Load documents from S3

Keep this section short. PMs need to know what documents are being used and whether the corpus is trustworthy.


In [ ]:
def list_docx_keys(bucket: str, prefix: str) -> list[str]:
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix)
    return sorted([
        obj["Key"] for obj in response.get("Contents", [])
        if obj["Key"].lower().endswith(".docx")
    ])


def read_docx_from_s3(bucket: str, key: str) -> str:
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()
    doc = DocxDocument(io.BytesIO(body))
    return "\n\n".join([p.text.strip() for p in doc.paragraphs if p.text.strip()])

keys = list_docx_keys(S3_BUCKET, S3_PREFIX)
print("Documents found:", len(keys))
print(keys[:10])

raw_documents = []
for key in keys:
    text = read_docx_from_s3(S3_BUCKET, key)
    if text.strip():
        raw_documents.append({"source": key, "text": text, "char_len": len(text)})

docs_df = pd.DataFrame(raw_documents)
display(docs_df[["source", "char_len"]])


## 2. Split documents into chunks

Use one simple chunking strategy. Do not compare multiple chunking methods in this shortened delivery.


In [ ]:
def chunk_with_overlap(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> list[str]:
    clean_text = re.sub(r"\s+", " ", text).strip()
    chunks = []
    start = 0
    while start < len(clean_text):
        end = start + chunk_size
        chunks.append(clean_text[start:end])
        if end >= len(clean_text):
            break
        start = max(0, end - overlap)
    return chunks

chunk_rows = []
for doc in raw_documents:
    for idx, chunk in enumerate(chunk_with_overlap(doc["text"]), start=1):
        chunk_rows.append({
            "source": doc["source"],
            "chunk_id": f"{doc['source']}::chunk_{idx:03d}",
            "chunk_text": chunk,
            "char_len": len(chunk),
        })

chunks_df = pd.DataFrame(chunk_rows)
if MAX_CHUNKS_TO_EMBED:
    chunks_df = chunks_df.head(MAX_CHUNKS_TO_EMBED).copy()

display(chunks_df.head())
print("Total chunks:", len(chunks_df))


## 3. Create embeddings and retrieve relevant chunks

This is where enterprise documents become searchable by meaning, not just keywords.


In [ ]:
def embed_text(text: str) -> list[float]:
    response = bedrock_runtime.invoke_model(
        modelId=BEDROCK_EMBEDDING_MODEL_ID,
        body=json.dumps({"inputText": text}),
        accept="application/json",
        contentType="application/json",
    )
    body = json.loads(response["body"].read())
    return body["embedding"]


def cosine_similarity(a, b) -> float:
    a = np.array(a)
    b = np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

chunk_embeddings = []
for text in chunks_df["chunk_text"]:
    chunk_embeddings.append(embed_text(text))

chunks_df["embedding"] = chunk_embeddings
print("Embeddings created:", len(chunk_embeddings))


In [ ]:
def retrieve(query: str, top_k: int = TOP_K) -> pd.DataFrame:
    query_embedding = embed_text(query)
    scored = chunks_df.copy()
    scored["score"] = scored["embedding"].apply(lambda e: cosine_similarity(query_embedding, e))
    return scored.sort_values("score", ascending=False).head(top_k)[["source", "chunk_id", "score", "chunk_text"]]

sample_question = "What is the leave policy?"
retrieved_df = retrieve(sample_question)
display(retrieved_df)


## 4. Generate grounded answers

The LLM is instructed to answer only from retrieved context and say when evidence is missing.


In [ ]:
def call_llm(prompt: str, max_tokens: int = 350) -> dict:
    start = time.time()
    response = bedrock_runtime.converse(
        modelId=BEDROCK_LLM_MODEL_ID,
        system=[{"text": "You are a careful enterprise RAG assistant. Use only the supplied context. If evidence is missing, say so."}],
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"temperature": 0.1, "topP": 0.9, "maxTokens": max_tokens},
    )
    return {
        "answer": response["output"]["message"]["content"][0]["text"],
        "usage": response.get("usage", {}),
        "latency_ms": round((time.time() - start) * 1000, 2),
    }


def rag_answer(question: str, top_k: int = TOP_K) -> dict:
    retrieved = retrieve(question, top_k=top_k)
    context = "\n\n".join([
        f"Source: {row.source}\nChunk: {row.chunk_text}"
        for row in retrieved.itertuples()
    ])
    prompt = f"""
Question:
{question}

Retrieved context:
{context}

Answer with:
1. Direct answer
2. Evidence used
3. Any limitation or missing evidence
"""
    llm_result = call_llm(prompt)
    return {"question": question, "answer": llm_result["answer"], "retrieved": retrieved, **llm_result}

result = rag_answer("What is the leave policy?")
print(result["answer"])
display(result["retrieved"][["source", "score", "chunk_text"]])


## 5. Golden-question evaluation

This replaces the separate learner activity notebook. Use only 3–5 questions in class.


In [ ]:
golden_questions = [
    {
        "question": "What is the leave policy?",
        "expected_answer_keywords": ["leave", "days", "approval"],
        "expected_source_hint": "Leave",
    },
    {
        "question": "What is the remote work policy?",
        "expected_answer_keywords": ["remote", "manager", "approval"],
        "expected_source_hint": "Remote",
    },
    {
        "question": "What is the company's moon mission policy?",
        "expected_answer_keywords": ["missing", "not available", "no evidence"],
        "expected_source_hint": "None",
    },
]

eval_rows = []
for item in golden_questions:
    output = rag_answer(item["question"])
    answer_lower = output["answer"].lower()
    retrieved_sources = " | ".join(output["retrieved"]["source"].astype(str).tolist())
    keyword_hits = sum(1 for kw in item["expected_answer_keywords"] if kw.lower() in answer_lower)
    source_hit = item["expected_source_hint"].lower() in retrieved_sources.lower() if item["expected_source_hint"] != "None" else True
    eval_rows.append({
        "question": item["question"],
        "keyword_hits": keyword_hits,
        "source_hit": source_hit,
        "answer_correct_quick_check": keyword_hits >= 1,
        "hallucination_risk": "Review" if keyword_hits == 0 else "Low",
        "top_sources": retrieved_sources,
        "answer": output["answer"],
    })

eval_df = pd.DataFrame(eval_rows)
display(eval_df)


## PM reflection checkpoint

Ask learners:

- Did retrieval find the right source?
- Did the answer stay grounded?
- What would you change: document quality, chunking, prompt, top_k, or evaluation set?
- What success metric would you put in the PRD?
